# Data Handling & Preprocessing — turning raw World Bank data into a model-ready dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/production-ai-engineering/blob/main/notebooks/hello_preprocessing_worldbank.ipynb)

**Quintrix Week 1 — Data Handling & Preprocessing (the 6-hr core).** Self-contained: the World Bank
dataset is embedded below, so this runs anywhere with **no internet and no data files**.

The pipeline, in the order a real engineer works it:
1. **Interrogate** — what do we have, and what's wrong with it?
2. **Missing values** — detect → decide (drop vs. impute)
3. **Outliers** — detect (IQR, z-score, boxplot) → decide (investigate, cap, drop)
4. **Normalization** — *log transform* (fix skew) **and** *feature scaling* (StandardScaler / MinMax)
5. **Feature engineering** — build columns a model can use
6. **Model-ready** — the payoff, and the link back to Day 2's linear regression

> **Why this matters:** a model is only as good as the data you feed it. This front half of the
> pipeline is ~80% of real AI work and where most of the judgment lives.

## 0. Setup + load the data (embedded — no download)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from io import StringIO
%matplotlib inline

# The World Bank 2021 dataset (168 countries) is embedded as text so this notebook is self-contained.
CSV = """country,income_group,gdp_per_capita,life_expectancy,electricity_pct,basic_water_pct,internet_pct,fertility_rate,under5_mortality,health_spend_pc
Afghanistan,LIC,356.496214115892,60.417,97.7,74.5545259961346,16.51429939,5.039,58.7,81.52112579
Angola,LMC,2266.96834861584,62.958,48.2,65.716257719435,34.55649948,5.304,53.2,66.94487
Albania,UMC,7242.45513146613,76.844,100.0,95.0815150513051,79.32371752,1.365,9.5,465.57043457
Andorra,HIC,42425.6996761083,82.331,100.0,100.0,93.89749908,1.058,2.8,3668.44750977
United Arab Emirates,HIC,44118.4936214315,79.083,100.0,100.0,100.0,1.162,5.6,2249.96313477
Armenia,UMC,4685.17997128509,72.2780487804878,100.0,99.4534764633698,78.61225758,1.7,11.0,595.83544922
Antigua and Barbuda,HIC,17348.6006987182,77.197,100.0,98.9226250605056,71.64530182,1.58,9.8,835.06512451
Australia,HIC,60758.9044397144,83.3,100.0,99.9697732704149,96.97219849,1.7,3.7,7092.13476563
Austria,HIC,53648.6437976214,81.190243902439,100.0,100.0,92.52915104,1.48,3.4,6554.14257813
Azerbaijan,UMC,5408.04535175023,70.999,100.0,95.0977447085139,85.99998954,1.52,20.5,250.9887085
Burundi,LIC,264.172844085398,62.101,10.2,65.0493033882129,6.777999878,5.078,52.3,23.11420059
Belgium,HIC,51658.238294958,81.790243902439,100.0,99.9999987625062,92.78867618,1.6,3.8,5847.17333984
Benin,LMC,1318.65158075198,59.61,42.0,68.6381245479094,28.20470047,4.707,82.2,38.45053101
Burkina Faso,LIC,895.535288442991,60.046,19.0,50.0752719507407,18.1522007,4.361,82.4,63.59171295
Bangladesh,LMC,2482.84917843357,71.103,99.0,98.4379436580879,38.91744466,2.173,30.9,56.09640121
Bulgaria,HIC,12966.8085197381,71.2121951219512,99.8,99.2363227360567,75.27144583,1.8,6.3,1109.10290527
Bahrain,HIC,27147.8083882286,78.083,100.0,99.9198990209829,100.0,1.852,8.2,1176.92102051
Bosnia and Herzegovina,UMC,7295.34378682258,74.567,100.0,96.1188407812551,75.67620138,1.49,6.4,697.07427979
Belarus,UMC,7489.71894713888,72.0714634146342,100.0,99.3932172039885,86.88841196,1.371,2.7,484.03930664
Belize,UMC,6142.98930557031,70.917,97.7,94.6148163127858,70.65370178,1.837,13.0,308.93920898
Bolivia,LMC,4010.76053686204,61.427,98.6,92.1056066930792,65.9773785,2.618,18.2,276.20880127
Brazil,UMC,7972.5366498646,73.038,99.5,99.4272648803619,80.68989314,1.638,14.7,768.27075195
Barbados,HIC,21084.4013935998,76.58,100.0,99.0674985116994,69.4756012,1.709,10.7,1391.49853516
Brunei Darussalam,HIC,31006.9636289785,74.869,100.0,100.0,97.62879944,1.778,10.3,665.87030029
Bhutan,LMC,3570.61257980872,72.448,100.0,99.1933487306505,85.63688389,1.488,20.0,161.46617126
Botswana,UMC,7807.88794570471,63.303,73.7,91.6110310251098,54.70069885,2.827,37.1,492.14276123
Canada,HIC,52886.6407813788,81.4514634146342,100.0,98.4271662446964,93.93710327,1.44,5.2,6511.46972656
Switzerland,HIC,96582.8687080693,83.7512195121951,100.0,99.9999966128277,95.56937154,1.52,4.0,11197.76464844
Chile,HIC,16216.1840860289,78.876,100.0,98.7780212070676,90.23280334,1.176,6.7,1576.69287109
China,UMC,12887.4357242941,78.117,100.0,94.6329735273297,73.05323517,1.117,7.0,670.25897217
Cote d'Ivoire,LMC,2455.98127641148,60.29,71.1,76.3229847618577,35.95069885,4.411,70.8,92.39588928
Cameroon,LMC,1672.32657340872,61.146,65.4,69.787122797367,38.89260101,4.468,71.9,72.59731293
"Congo, Dem. Rep.",LIC,595.743074403567,60.044,20.8,35.3236207810538,16.1595993,6.156,92.9,21.58848572
"Congo, Rep.",LMC,2516.16255145432,64.192,49.7,73.4884211673414,30.02739906,4.28,43.0,78.60149384
Colombia,UMC,6222.62164397614,72.698,100.0,96.5223081416627,73.02837449,1.68,12.7,568.84130859
Comoros,LMC,1587.72988658008,65.201,87.9,85.368182871054,30.68120003,3.999,42.6,100.50016785
Cabo Verde,UMC,3971.44409203423,74.424,95.5,89.2475125078835,70.28250122,1.554,12.7,262.42272949
Costa Rica,HIC,12962.2716075659,78.05,100.0,99.8118715066538,82.74901607,1.353,9.6,977.14831543
Cyprus,HIC,32937.8359375,80.573,100.0,99.737372137771,90.75951564,1.392,3.7,2947.81494141
Czechia,HIC,27696.4619527544,77.2219512195122,100.0,98.7326235754859,82.67062427,1.83,2.8,2538.03344727
Germany,HIC,52349.2459994422,80.790243902439,100.0,99.9999988425523,91.43060973,1.58,3.7,6653.58349609
Djibouti,LMC,3025.91126413549,63.83,65.4,78.9636292425205,61.41120148,2.699,54.0,86.88237762
Dominica,UMC,8376.40414141113,69.832,100.0,96.6813469673205,78.07949829,1.484,34.4,521.41455078
Denmark,HIC,69340.7334922138,81.4048780487805,100.0,100.0,98.8658506,1.72,3.7,7429.99023438
Dominican Republic,UMC,8527.0754858205,71.757,98.1,97.3054739452924,85.24298638,2.294,33.0,433.89172363
Algeria,UMC,4160.55926736925,75.208,99.8,92.273231858432,72.21240234,2.87,22.6,208.93911743
Ecuador,UMC,6061.32350181711,72.746,100.0,91.6679688457759,69.1147995,1.909,13.4,507.65136719
"Egypt, Arab Rep.",LMC,3827.35415374308,68.976,100.0,97.5013675608953,72.05599976,2.75,24.0,176.93664551
Spain,HIC,30799.4775937775,83.1780487804878,100.0,99.9253882857533,93.89752157,1.18,3.2,3188.8515625
Estonia,HIC,27951.1659277744,76.9439024390244,100.0,99.9999994683624,90.97970174,1.61,2.3,2091.15649414
Ethiopia,LIC,893.009835441784,65.33,54.2,51.1547750099921,16.6981031,4.177,70.4,26.06931114
Finland,HIC,53099.1351400148,81.8853658536585,100.0,99.9999969157891,92.80806082,1.46,2.4,5261.90966797
Fiji,UMC,4536.73312110335,64.904,92.1,95.4534634943263,70.56800079,2.332,28.3,331.58413696
France,HIC,43725.0999521245,82.3243902439024,100.0,100.0,86.09548575,1.83,4.3,5364.89746094
Gabon,UMC,8181.40914134398,67.072,91.8,87.9954718713657,65.76809692,3.776,35.8,228.46922302
United Kingdom,HIC,47695.6491523474,80.6473170731707,100.0,100.0,96.19650269,1.583,4.4,5645.40136719
Georgia,UMC,5083.60695484577,71.635,100.0,96.877911792559,76.4427093,1.98,9.2,524.32415771
Ghana,LMC,2445.18662529344,64.286,86.3,86.841252750433,68.6,3.492,40.3,94.40388489
Guinea,LMC,1244.96550477914,59.373,46.8,70.7443560609114,28.61330032,4.399,100.3,44.10476303
"Gambia, The",LIC,781.890925169531,63.847,63.7,85.696311673001,41.67309952,4.165,47.0,29.77283478
Guinea-Bissau,LIC,926.54619187706,61.653,35.8,61.5510835646003,23.3519001,3.998,74.3,70.21189117
Greece,HIC,20652.89659591,80.0829268292683,100.0,100.0,78.49426371,1.43,4.0,1846.37304688
Guatemala,UMC,4912.62240414681,67.858,97.9,92.5346655416111,50.84164945,2.388,22.7,361.51858521
Guyana,HIC,9860.87014834386,64.323,92.9,95.3488391542383,71.65090179,2.462,27.8,457.86633301
Honduras,LMC,2735.14751503977,69.492,94.1,94.6929450618276,51.95159912,2.548,16.5,253.63349915
Croatia,HIC,17788.7601163765,76.4243902439025,100.0,99.9022621135501,81.25396608,1.62,4.6,1423.52905273
Haiti,LMC,1824.6798742577,62.613,47.2,71.2216853162772,43.37919998,2.748,58.1,59.89406204
Hungary,HIC,19030.628130329,74.0658536585366,100.0,99.9722511315841,88.64083466,1.63,4.0,1392.97460938
Indonesia,UMC,4287.17313995098,67.452,99.2,89.4041137814594,62.10447842,2.166,19.9,158.89761353
India,LMC,2239.61384367482,67.282,99.6,93.760281159736,49.2580986,2.014,31.2,75.54971313
Ireland,HIC,103783.446284479,82.2536585365854,100.0,97.0473662537478,93.50610352,1.72,3.6,6754.07519531
"Iran, Islamic Rep.",UMC,4605.14881318992,73.752,100.0,97.5146314590197,78.5957366,1.709,12.6,196.66934204
Iraq,UMC,4868.49431080783,70.704,100.0,98.3711569251332,65.00219727,3.358,24.0,251.59538269
Iceland,HIC,70425.4064277273,83.1658536585366,100.0,100.0,99.68701957,1.82,2.9,6722.421875
Israel,HIC,52258.4693499398,82.5,100.0,100.0,90.29689758,3.0,3.5,4294.18847656
Italy,HIC,36852.5425414951,82.6463414634146,100.0,99.9170340740928,74.8623274,1.25,2.9,3414.57299805
Jamaica,UMC,5625.67806271236,69.085,100.0,88.5656727695016,82.36071561,1.378,18.6,413.36227417
Jordan,UMC,4581.71967230365,74.204,99.9,99.200324026186,85.999997,2.735,14.1,322.01589966
Japan,HIC,41580.739040705,84.4456097560976,100.0,99.1133201720243,82.91407928,1.3,2.4,4843.94433594
Kazakhstan,UMC,9983.60103576902,70.131,100.0,97.2275162757939,90.9239507,3.32,10.0,391.81536865
Kenya,LMC,2061.35622089594,61.225,76.5,63.1426684243247,26.96290016,3.312,41.3,94.20249176
Kyrgyz Republic,LMC,1349.99730650542,71.9,99.7,93.250254637138,77.11769867,2.89,17.5,72.58779144
Cambodia,LMC,2167.40324198499,69.301,82.5,79.7640004029773,60.76269913,2.664,20.8,137.38909912
Kiribati,LMC,2223.54147866465,64.148,92.8,77.5479842310266,67.08760071,3.219,57.2,263.09573364
"Korea, Rep.",HIC,37518.4635305697,83.5268292682927,100.0,99.8371059639777,97.57132698,0.808,3.0,3125.82714844
Kuwait,HIC,34018.6342592913,78.7365853658537,100.0,100.0,99.7,2.148,8.7,1829.13134766
Lao PDR,LMC,2526.05105018529,67.807,100.0,87.5242448127318,62.0,2.494,33.2,68.61697388
Lebanon,LMC,4045.37391066234,73.65,100.0,92.6,82.65599823,2.283,15.7,189.8581543
Liberia,LIC,667.96610514319,61.166,29.8,76.8629734925215,26.38229942,4.089,88.0,88.54867554
Libya,UMC,4935.82787349801,72.062,70.2,95.5357202334962,77.438797,2.462,10.8,298.84338379
St. Lucia,UMC,10481.1233971786,69.119,100.0,97.6128621047241,63.06330109,1.399,17.9,654.8918457
Sri Lanka,UMC,3996.96240531688,76.278,100.0,89.5988356123165,44.45308724,1.995,6.6,202.75497437
Lesotho,LMC,1066.58645143823,54.209,50.4,76.300316439538,45.05670166,2.769,64.0,115.81448364
Lithuania,HIC,23882.8668916753,74.0365853658537,100.0,99.2383026544667,86.93054425,1.36,3.6,1844.33154297
Luxembourg,HIC,134965.815442152,82.5975609756098,100.0,100.0,98.66086333,1.38,2.3,7632.57275391
Latvia,HIC,20261.8887788916,72.9804878048781,100.0,98.6663184077259,91.17964988,1.57,3.3,1897.95275879
Morocco,LMC,3785.93627929688,73.385,100.0,89.0414569298463,88.13031937,2.288,17.6,209.57344055
Monaco,HIC,223823.327192464,85.113,100.0,100.0,99.07759857,2.122,2.9,8225.38769531
Moldova,UMC,5274.60582195352,68.991,100.0,91.4092054171204,74.91899872,1.751,15.8,403.87823486
Madagascar,LIC,483.469463351195,62.516,35.1,52.9732968597974,13.74590015,4.104,65.0,17.17938805
Maldives,UMC,10176.143882547,78.052,100.0,99.675443163466,81.95760345,1.599,6.4,901.09185791
Mexico,UMC,10314.0506733708,69.75,100.0,99.0090934864526,75.62650476,1.97,14.2,606.13415527
Marshall Islands,UMC,6315.31172190393,65.421,99.8,87.579321179184,63.97259903,2.964,29.4,782.16351318
North Macedonia,UMC,7620.80296977508,73.2463414634146,100.0,98.1308494044727,81.873703,1.6,5.1,636.65447998
Mali,LIC,1027.27345365357,59.116,53.4,82.2652506105548,33.95650101,5.779,83.6,30.55883217
Malta,HIC,38078.0064279067,82.2585365853659,100.0,99.9999977128228,87.46977742,1.13,5.8,3641.03833008
Montenegro,UMC,9315.86435643142,73.8243902439024,99.8,99.2830138723667,82.21969752,1.75,2.8,1024.2565918
Mongolia,UMC,4517.61578018785,71.6451219512195,100.0,83.761628489025,81.60682285,2.8,14.3,330.03753662
Mozambique,LIC,509.907829475551,60.268,31.5,60.34998629028,16.35759926,4.905,65.5,44.09894943
Mauritania,LMC,1947.47413907209,66.762,47.7,73.3049522964887,35.63660049,4.848,40.7,73.6309433
Mauritius,UMC,9177.71195973013,73.680243902439,99.6,99.8537467482534,71.59999847,1.42,15.7,593.77947998
Malawi,LIC,617.446994451007,64.782,14.2,70.6341661895421,15.30300045,3.791,50.0,46.41017532
Malaysia,UMC,10903.1037480772,73.917,100.0,97.7858740262041,96.7514278,1.555,8.2,476.94366455
Namibia,LMC,4412.8355693052,60.85,55.2,85.3713640915778,60.25999832,3.303,43.2,449.95172119
Niger,LIC,609.585703670137,59.542,18.6,51.4037852396431,12.45689964,6.2,118.0,35.39406204
Nigeria,LMC,2787.48779220991,53.455,59.5,77.8385518833185,35.50979996,4.636,117.6,81.87322235
Netherlands,HIC,60141.9880911492,81.309756097561,100.0,99.9999991132456,92.05298013,1.62,4.0,6718.88964844
Norway,HIC,96442.5552173002,83.1634146341463,100.0,99.999996677432,99.0000036,1.55,2.4,9163.13867188
Nepal,LMC,1252.75076731986,68.385,89.9,92.0505329995059,40.98910141,2.025,28.3,69.1880188
Naoero,HIC,12748.1701249353,61.331,100.0,97.9304771849756,80.29499817,3.425,10.0,1636.45788574
New Zealand,HIC,49902.154346032,82.2073170731707,100.0,99.9999956407702,91.99060059,1.65,5.0,4749.04199219
Oman,HIC,19403.4600160211,75.969,100.0,92.3097194594894,95.23880005,2.588,10.6,843.01953125
Panama,HIC,15509.8069109784,77.004,95.3,94.6436730326195,66.04270172,2.152,15.2,1417.1541748
Peru,UMC,6828.53422535638,71.596,95.6,93.9287808522473,71.10782643,2.029,14.1,455.3560791
Philippines,UMC,3484.38593882819,66.675,97.5,94.6741026690457,66.90779877,1.95,27.9,208.192276
Papua New Guinea,LMC,2607.97769973161,64.358,20.9,50.3501518830702,16.55960083,3.215,43.2,61.02529907
Poland,HIC,18635.5104895754,75.3512195121951,100.0,91.710929872632,85.37490906,1.33,4.4,1183.30236816
Portugal,HIC,24711.4510062111,81.3780487804878,100.0,99.2456792845516,82.30902804,1.35,3.3,2764.04296875
Paraguay,UMC,5974.9057185863,68.108,100.0,99.3305060939918,77.01838621,2.469,18.0,480.11322021
Qatar,HIC,71751.8831257169,81.083,100.0,99.9828259022685,99.65280151,1.605,6.2,1847.00231934
Romania,HIC,14908.0412813921,72.809756097561,100.0,100.0,83.5904265,1.81,6.4,963.01287842
Russian Federation,HIC,12425.029296875,69.900243902439,100.0,96.8045794762189,88.21384578,1.505,5.5,881.74395752
Rwanda,LIC,842.581492947672,66.85,48.7,59.6381227763257,24.4416008,3.841,40.6,51.64677429
Saudi Arabia,HIC,31920.7653655643,77.089,100.0,99.0455148837977,100.0,2.17,6.7,1654.74353027
Senegal,LMC,1598.10676952274,66.868,68.0,84.636074800772,55.04040146,3.936,42.2,71.12162781
Singapore,HIC,80884.8565369236,83.0926829268293,100.0,100.0,96.92468156,1.12,2.4,4050.81054688
Solomon Islands,LMC,2043.44413170658,68.89,76.3,71.2454871530386,26.03790092,3.694,21.9,98.80261993
Sierra Leone,LIC,885.396401613729,60.26,27.5,64.5803175608776,18.65390015,3.978,100.6,44.90049362
El Salvador,UMC,4642.60743101342,69.941,97.9,90.421388926903,57.26380157,1.803,11.2,470.14025879
San Marino,HIC,54168.9744311193,83.539,100.0,100.0,81.66999817,1.125,1.4,4033.1027832
"Somalia, Fed. Rep.",LIC,549.056994756254,55.703,49.3,67.9535994970856,23.87479973,6.353,111.0,15.01879692
Serbia,UMC,9680.52798074468,72.780487804878,100.0,95.4256771978758,81.16588116,1.52,5.4,889.09753418
Sao Tome and Principe,LMC,2362.58827248102,68.026,78.5,77.1104795429933,54.94950104,3.759,15.3,187.06524658
Slovak Republic,HIC,22123.3342114545,74.6146341463415,100.0,99.7874622291047,88.92560523,1.63,6.0,1684.51550293
Slovenia,HIC,29192.8401933019,80.6756097560976,100.0,99.5845624791202,89.00399524,1.64,2.3,2779.56591797
Sweden,HIC,60647.5416365347,83.0560975609756,100.0,99.7393684296399,94.6703163,1.67,2.5,6894.20166016
Eswatini,LMC,3984.00229174306,58.228,82.9,76.7260694607932,53.99530029,2.841,47.9,297.72213745
Seychelles,HIC,14982.9111487767,73.3975609756097,100.0,97.2072138644351,81.99530029,2.38,14.7,629.85974121
Chad,LIC,946.358500208374,53.136,11.3,51.8991024750567,9.653699875,6.255,107.3,37.59156799
Togo,LMC,1076.29520076949,61.337,55.7,62.4040009740747,30.33839989,4.323871,62.0,52.72237015
Thailand,UMC,7055.18760721422,77.606,100.0,99.9512350608319,85.26956666,1.234,9.8,363.72845459
Tajikistan,LMC,896.748053372182,69.594,99.6,87.724619438918,36.70560074,3.17,29.8,75.10315704
Timor-Leste,LMC,2684.92673717299,66.199,100.0,82.6385824192743,35.04199982,2.892,52.0,132.07870483
Tonga,UMC,4922.8015864044,72.13,100.0,98.7654258730206,57.549,3.198,10.4,348.79745483
Trinidad and Tobago,HIC,17712.5674104655,71.111,100.0,98.8610651706074,84.36576986,1.559,20.4,1170.93859863
Tunisia,LMC,3906.93926319391,72.893,99.9,95.5312981932344,68.31552226,1.8,15.0,285.25247192
Turkiye,UMC,9981.76398411336,75.722,100.0,95.9614082696356,81.40843745,1.71,10.6,431.33978271
Tanzania,LMC,1159.85656738281,66.131,42.7,59.9769893426504,22.87540054,4.734,41.7,37.53121185
Uganda,LIC,882.791717542504,66.45,45.2,57.9452003284678,10.0,4.51,50.2,51.03484726
Ukraine,UMC,4775.94580078125,71.631,100.0,93.4969374195088,79.21829353,1.148,8.2,369.90487671
Uruguay,HIC,17881.8119616753,75.434,100.0,99.5860335482368,87.64440155,1.446,7.3,1594.98059082
United States,HIC,71441.2319805947,76.3292682926829,100.0,99.894943997231,91.2684021,1.664,6.5,12080.07324219
Uzbekistan,LMC,2370.36480555669,71.681,99.9,96.2897455645648,76.59043014,3.273,14.6,156.44789124
"Venezuela, RB",LMC,2004.93572921136,71.536,100.0,93.5087058493878,74.11219788,2.105,24.2,162.43952942
Viet Nam,UMC,3704.1935590038,74.145,100.0,96.9270863481492,74.20999985,1.942,18.9,168.09350586
South Africa,UMC,6828.75624590017,62.01,89.3,90.9231026021655,74.96859741,2.248,34.0,592.9465332
Zambia,LMC,1127.16077872857,62.363,46.7,70.2228298527718,14.82190037,4.246,50.4,74.83802795
Zimbabwe,LMC,2613.6167414057,60.135,49.0,67.162605340134,32.5923996,3.765,62.4,60.89874268
"""
df = pd.read_csv(StringIO(CSV))
print("shape:", df.shape)
df.head()

## 1. Interrogate — always look before you touch
Three commands you run on *every* new dataset: `.info()` (types + non-null counts), `.describe()`
(the numbers' distribution), `.isna().sum()` (where the holes are).

In [ ]:
df.info()
print("\nMissing per column:\n", df.isna().sum().to_string())
df.describe().round(1)

## 2. Missing values — detect → decide
This extract was pre-cleaned, so it has no holes. **Real** World Bank exports always do — so we'll
inject a few to practice on (this is a teaching move, stated honestly). The skill isn't *filling* holes,
it's **choosing** how: drop the rows, or impute (fill) with the median? The choice changes your answer.

In [ ]:
# Simulate real-world missingness: knock out ~8% of two columns (deterministic, no RNG).
work = df.copy()
work.loc[work.index % 13 == 0, "internet_pct"] = np.nan     # ~13 holes
work.loc[work.index % 17 == 0, "health_spend_pc"] = np.nan  # ~10 holes
print("Holes introduced:\n", work[["internet_pct","health_spend_pc"]].isna().sum().to_string())

# Option A -- DROP any row with a hole: simple, but you lose whole countries.
dropped = work.dropna()
print(f"\nDrop rows  -> {len(dropped)} countries kept (lost {len(work)-len(dropped)})")

# Option B -- IMPUTE with the median: keep every country, fill the gap with a typical value.
imputed = work.copy()
for col in ["internet_pct","health_spend_pc"]:
    imputed[col] = imputed[col].fillna(imputed[col].median())
print(f"Impute     -> {len(imputed)} countries kept (0 lost)")
print("\nRule of thumb: DROP if holes are few and rows are cheap; IMPUTE if every row matters")
print("(here, every country matters -> we impute). Never ignore them -- most models refuse NaNs.")
df = imputed  # carry the imputed frame forward

## 3. Outliers — detect → decide
An outlier is a value far from the rest. Two ways to catch them, then the real question: *is it a
data error (drop it) or a real signal (keep it — it may be the whole story)?*

In [ ]:
# Method 1 -- IQR rule: outlier if outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR].
col = "gdp_per_capita"
q1, q3 = df[col].quantile([0.25, 0.75]); iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
iqr_out = df[(df[col] < lo) | (df[col] > hi)]
print(f"IQR flags {len(iqr_out)} GDP outliers (above ${hi:,.0f}):")
print(iqr_out.nlargest(5, col)[["country", col]].to_string(index=False))

# Method 2 -- z-score: how many standard deviations from the mean (|z| > 3 is a common cutoff).
z = (df[col] - df[col].mean()) / df[col].std()
print(f"\nz-score flags {int((z.abs()>3).sum())} countries with |z|>3")

print("\nDECIDE: these are rich, small economies (Luxembourg, Qatar...) -- a REAL signal, not errors.")
print("So we KEEP them. (Section 4's log transform tames their pull WITHOUT deleting data.)")

In [ ]:
# A boxplot makes outliers visible instantly -- the dots beyond the whiskers.
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].boxplot(df["gdp_per_capita"]); ax[0].set_title("GDP per capita -- raw (long tail of rich outliers)")
ax[0].set_ylabel("USD")
ax[1].boxplot(np.log(df["gdp_per_capita"])); ax[1].set_title("GDP per capita -- log (outliers pulled in)")
ax[1].set_ylabel("ln(USD)")
plt.tight_layout(); plt.show()

## 4. Normalization — two different things people call by one name
- **Log transform** — fixes *skew* (a few huge values stretching the scale). Used it in Day 2 on GDP.
- **Feature scaling** — puts *different columns on a common range* so no single one dominates just by
  its units. Two standard tools: **StandardScaler** (mean 0, std 1 — the "z-score") and **MinMaxScaler**
  (squash to 0–1). Models that use distances or gradient descent (KNN, SVM, neural nets, logistic) need this.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

df["log_gdp"] = np.log(df["gdp_per_capita"])   # log transform (skew fix)

feats = ["life_expectancy", "internet_pct", "under5_mortality", "log_gdp"]
print("BEFORE scaling -- wildly different ranges (life~80, internet~0-100, log_gdp~6-12):")
print(df[feats].describe().loc[["min","max"]].round(1).to_string())

std = pd.DataFrame(StandardScaler().fit_transform(df[feats]), columns=feats)
print("\nAfter StandardScaler -- every column mean~0, std~1:")
print(std.describe().loc[["mean","std"]].round(2).to_string())

mm = pd.DataFrame(MinMaxScaler().fit_transform(df[feats]), columns=feats)
print("\nAfter MinMaxScaler -- every column squashed to [0, 1]:")
print(mm.describe().loc[["min","max"]].round(2).to_string())
print("\nWHY: without scaling, 'internet_pct' (0-100) would swamp 'log_gdp' (6-12) purely by magnitude.")

## 5. Feature engineering — build columns a model can use
Raw columns are rarely the best inputs. Domain knowledge enters here: combine, ratio, or bin existing
columns into something more predictive.

In [ ]:
# (a) a ratio: health spending as a share of income -- more comparable across countries than raw dollars
df["health_share"] = df["health_spend_pc"] / df["gdp_per_capita"]
# (b) a binary flag: is this a high-connectivity country?
df["high_internet"] = (df["internet_pct"] > 50).astype(int)
# (c) one-hot encode the categorical income_group so a model can read it (LIC/LMC/UMC/HIC -> columns)
df = pd.get_dummies(df, columns=["income_group"], prefix="inc")

print("New engineered columns:")
print(df[["country","health_share","high_internet"]].head(4).to_string(index=False))
print("\nOne-hot income columns:", [c for c in df.columns if c.startswith("inc_")])

## 6. Model-ready — the payoff
The dataset is now clean, scaled-ready, and feature-rich. This is exactly the tidy table Day 2's
linear regression assumed it was handed — except now *you built it*. That front half is the job.

In [ ]:
model_cols = ["log_gdp","life_expectancy","internet_pct","under5_mortality",
              "health_share","high_internet"] + [c for c in df.columns if c.startswith("inc_")]
ready = df[["country"] + model_cols]
print("Model-ready shape:", ready.shape, "-- no NaNs:", int(ready[model_cols].isna().sum().sum())==0)
print("\nDay 2 recap: life_expectancy = 32.3 + 4.4 * log_gdp -- the 'log_gdp' column above is that feature,")
print("now produced by a real preprocessing pipeline instead of handed to you clean.")
ready.head()

## Takeaway
- **Interrogate first** (`.info` / `.describe` / `.isna`) — never model blind.
- **Missing values:** drop vs. impute is a *judgment* call — every row matters here, so we imputed.
- **Outliers:** detect with IQR / z-score / boxplot, then decide — real signal (keep) vs. error (drop).
- **Normalization ≠ one thing:** log fixes *skew*; StandardScaler/MinMax fix *scale* across columns.
- **Feature engineering** is where domain knowledge becomes predictive columns.
- The output is a **model-ready** table — the thing every algorithm in Weeks 2–3 assumes it's given.